# Pretrained Activation Experiment

**Research question:** If we train activation functions from scratch based on good weights, and then train *random* weights with those pretrained activations — does it reduce training time for weights?

## Experimental Pipeline (3 Phases)

**Phase 1 – Baseline:** Train weights with fixed ReLU activation.
$$\min_W \; \mathcal{L}(f(X; W, \text{ReLU}))$$
Produces $W^*$ (trained weights) and records epochs/time to converge.

**Phase 2 – Learn activations on top of good weights:**
Take $W^*$ from Phase 1, replace ReLU with $\text{DynamicReLU}(a, b)$, freeze weights, and solve:
$$\min_{a, b} \; \mathcal{L}(f(X; W^*, \text{DynamicReLU}(a, b)))$$
Produces $a^*, b^*$ (pretrained activation parameters).

**Phase 3 – Train random weights with pretrained activations:**
Initialize fresh random weights $W'$, plug in $a^*, b^*$ from Phase 2, freeze activations, and solve:
$$\min_{W'} \; \mathcal{L}(f(X; W', \text{DynamicReLU}(a^*, b^*)))$$
Records epochs/time to converge and compares with Phase 1.

**Questions:**
1. Will the final weights be different?
2. Did weight training finish earlier with pretrained activations?

## Setup & Imports

In [ ]:
import sys
import os
import copy
import time

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from scipy import stats

from activations import DynamicReLU
from layers import Dense
from mlp import MLP, MLPConfig
from mlp_trainer import MLPTrainer, CrossEntropyLoss, TrainingHistory
from data_utils import DatasetConfig, DataManager

print("Imports successful!")

## Configuration

In [ ]:
N_SEEDS = 20
SEEDS = [42, 123, 456, 789, 1001, 1111, 2222, 3333, 4444, 5555,
         6666, 7777, 8888, 9999, 1010, 2020, 3030, 4040, 5050, 6060]
HIDDEN_DIMS = [256, 128]
EPOCHS = 30
ACTIVATION_EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 0.01
PATIENCE = 10

DATASETS = ['mnist', 'fashion_mnist', 'cifar10']

print(f"Seeds: {N_SEEDS} | Architecture: {HIDDEN_DIMS} | Epochs: {EPOCHS}")
print(f"Activation epochs: {ACTIVATION_EPOCHS} | LR: {LEARNING_RATE}")
print(f"Datasets: {DATASETS}")

## Helper Utilities

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two flattened weight vectors."""
    a_flat = a.ravel()
    b_flat = b.ravel()
    denom = np.linalg.norm(a_flat) * np.linalg.norm(b_flat)
    if denom < 1e-12:
        return 0.0
    return float(np.dot(a_flat, b_flat) / denom)


def l2_distance(a: np.ndarray, b: np.ndarray) -> float:
    """L2 (Euclidean) distance between two flattened weight vectors."""
    return float(np.linalg.norm(a.ravel() - b.ravel()))


def extract_all_weights(model: MLP) -> np.ndarray:
    """Concatenate all layer weights and biases into a single flat vector."""
    parts = []
    for layer in model.layers:
        if isinstance(layer, Dense):
            parts.append(layer.weights.ravel())
            parts.append(layer.bias.ravel())
    return np.concatenate(parts)


def extract_activation_params(model: MLP) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Extract (a, b) activation parameters from each Dense layer with DynamicReLU."""
    params = []
    for layer in model.layers:
        if isinstance(layer, Dense) and isinstance(layer.activation, DynamicReLU):
            a = np.copy(layer.activation._a) if isinstance(layer.activation._a, np.ndarray) else np.array([layer.activation._a])
            b = np.copy(layer.activation._b) if isinstance(layer.activation._b, np.ndarray) else np.array([layer.activation._b])
            params.append((a, b))
    return params


def inject_activation_params(model: MLP, params: List[Tuple[np.ndarray, np.ndarray]]) -> None:
    """Inject pretrained (a, b) activation parameters into DynamicReLU layers."""
    idx = 0
    for layer in model.layers:
        if isinstance(layer, Dense) and isinstance(layer.activation, DynamicReLU):
            a_val, b_val = params[idx]
            if layer.activation.per_neuron:
                layer.activation._a = np.copy(a_val)
                layer.activation._b = np.copy(b_val)
            else:
                layer.activation._a = float(a_val.ravel()[0])
                layer.activation._b = float(b_val.ravel()[0])
            idx += 1

## Training Functions

In [ ]:
def train_weights_only(
    model: MLP, X_train: np.ndarray, y_train: np.ndarray,
    X_val: np.ndarray, y_val: np.ndarray, trainer: MLPTrainer,
) -> TrainingHistory:
    """Train only network weights; activation parameters are frozen."""
    start_time = time.time()
    num_classes = model.config.output_dim
    y_train_oh = trainer._to_onehot(y_train, num_classes)
    y_val_oh = trainer._to_onehot(y_val, num_classes)

    history = TrainingHistory(
        train_losses=[], train_accuracies=[],
        val_losses=[], val_accuracies=[],
        epochs_trained=0, training_time=0.0,
    )

    best_val_loss = float('inf')
    patience_counter = 0
    n = len(X_train)

    for epoch in range(trainer.epochs):
        model.train()
        idx = np.random.permutation(n)
        X_shuf, y_shuf = X_train[idx], y_train_oh[idx]

        epoch_losses = []
        for start in range(0, n, trainer.batch_size):
            end = min(start + trainer.batch_size, n)
            Xb, yb = X_shuf[start:end], y_shuf[start:end]
            preds = model.forward(Xb)
            loss = CrossEntropyLoss.compute(preds, yb)
            epoch_losses.append(loss)
            grad = CrossEntropyLoss.gradient(preds, yb)
            model.backward(grad, update_activation=False, update_weights=True)

        train_loss = float(np.mean(epoch_losses))
        train_acc = float(trainer._compute_accuracy(model, X_train, y_train))
        history.train_losses.append(train_loss)
        history.train_accuracies.append(train_acc)

        model.eval()
        val_preds = model.forward(X_val)
        val_loss = float(CrossEntropyLoss.compute(val_preds, y_val_oh))
        val_acc = float(trainer._compute_accuracy(model, X_val, y_val))
        history.val_losses.append(val_loss)
        history.val_accuracies.append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= trainer.early_stopping_patience:
                if trainer.verbose:
                    print(f"    Early stopping at epoch {epoch + 1}")
                break

        if trainer.verbose and (epoch + 1) % trainer.print_every == 0:
            print(f"    Epoch {epoch+1:>3}/{trainer.epochs} | Loss {train_loss:.4f} | Train {train_acc:.4f} | Val {val_acc:.4f}")

        history.epochs_trained = epoch + 1

    history.training_time = time.time() - start_time
    return history


def train_activations_only(
    model: MLP, X_train: np.ndarray, y_train: np.ndarray,
    X_val: np.ndarray, y_val: np.ndarray, trainer: MLPTrainer,
    epochs: int = 100,
) -> TrainingHistory:
    """Train only learnable activation parameters; weights are frozen."""
    start_time = time.time()
    num_classes = model.config.output_dim
    y_train_oh = trainer._to_onehot(y_train, num_classes)
    y_val_oh = trainer._to_onehot(y_val, num_classes)

    history = TrainingHistory(
        train_losses=[], train_accuracies=[],
        val_losses=[], val_accuracies=[],
        epochs_trained=0, training_time=0.0,
    )

    best_val_loss = float('inf')
    patience_counter = 0
    n = len(X_train)

    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(n)
        X_shuf, y_shuf = X_train[idx], y_train_oh[idx]

        epoch_losses = []
        for start in range(0, n, trainer.batch_size):
            end = min(start + trainer.batch_size, n)
            Xb, yb = X_shuf[start:end], y_shuf[start:end]
            preds = model.forward(Xb)
            loss = CrossEntropyLoss.compute(preds, yb)
            epoch_losses.append(loss)
            grad = CrossEntropyLoss.gradient(preds, yb)
            model.backward(grad, update_activation=True, update_weights=False)

        train_loss = float(np.mean(epoch_losses))
        train_acc = float(trainer._compute_accuracy(model, X_train, y_train))
        history.train_losses.append(train_loss)
        history.train_accuracies.append(train_acc)

        model.eval()
        val_preds = model.forward(X_val)
        val_loss = float(CrossEntropyLoss.compute(val_preds, y_val_oh))
        val_acc = float(trainer._compute_accuracy(model, X_val, y_val))
        history.val_losses.append(val_loss)
        history.val_accuracies.append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= trainer.early_stopping_patience:
                if trainer.verbose:
                    print(f"    [Activation] Early stopping at epoch {epoch + 1}")
                break

        if trainer.verbose and (epoch + 1) % trainer.print_every == 0:
            print(f"    [Act] Epoch {epoch+1:>3}/{epochs} | Loss {train_loss:.4f} | Train {train_acc:.4f} | Val {val_acc:.4f}")

        history.epochs_trained = epoch + 1

    history.training_time = time.time() - start_time
    return history

## Model Builders

In [ ]:
def build_fixed_relu_model(input_dim, hidden_dims, output_dim, lr):
    """Build an MLP with standard (fixed) ReLU hidden activations."""
    config = MLPConfig(
        input_dim=input_dim, hidden_dims=hidden_dims, output_dim=output_dim,
        hidden_activation="relu", output_activation="softmax",
        learning_rate=lr, weight_init="he",
    )
    return MLP(config)


def build_dynamic_relu_model(input_dim, hidden_dims, output_dim, lr, activation_lr, per_neuron=True):
    """Build an MLP with DynamicReLU hidden activations."""
    config = MLPConfig(
        input_dim=input_dim, hidden_dims=hidden_dims, output_dim=output_dim,
        hidden_activation="dynamic_relu", output_activation="softmax",
        learning_rate=lr, activation_lr=activation_lr,
        per_neuron_activation=per_neuron, weight_init="he",
    )
    return MLP(config)


def copy_weights_into(dst: MLP, src: MLP) -> None:
    """Copy weights and biases from src into dst (layer-by-layer)."""
    for d_layer, s_layer in zip(dst.layers, src.layers):
        if isinstance(d_layer, Dense) and isinstance(s_layer, Dense):
            d_layer.weights = s_layer.weights.copy()
            d_layer.bias = s_layer.bias.copy()

## Phase Result Data Classes

In [ ]:
@dataclass
class PhaseResult:
    """Stores per-phase metrics."""
    name: str
    epochs_trained: int
    training_time: float
    final_train_acc: float
    final_val_acc: float
    best_val_acc: float
    train_losses: List[float] = field(default_factory=list)
    val_losses: List[float] = field(default_factory=list)


@dataclass
class AggregatedPhaseResult:
    """Aggregated results across multiple seeds."""
    name: str
    epochs: List[int] = field(default_factory=list)
    times: List[float] = field(default_factory=list)
    val_accs: List[float] = field(default_factory=list)
    best_val_accs: List[float] = field(default_factory=list)
    train_accs: List[float] = field(default_factory=list)

    @property
    def epochs_mean(self): return float(np.mean(self.epochs))
    @property
    def epochs_std(self): return float(np.std(self.epochs))
    @property
    def time_mean(self): return float(np.mean(self.times))
    @property
    def time_std(self): return float(np.std(self.times))
    @property
    def val_acc_mean(self): return float(np.mean(self.val_accs))
    @property
    def val_acc_std(self): return float(np.std(self.val_accs))
    @property
    def best_val_mean(self): return float(np.mean(self.best_val_accs))
    @property
    def best_val_std(self): return float(np.std(self.best_val_accs))
    @property
    def train_acc_mean(self): return float(np.mean(self.train_accs))
    @property
    def train_acc_std(self): return float(np.std(self.train_accs))

## Core 3-Phase Experiment

In [ ]:
def run_experiment(
    X_train, y_train, X_val, y_val,
    hidden_dims=None, weight_lr=0.01, activation_lr=0.01,
    weight_epochs=30, activation_epochs=30, batch_size=128,
    patience=10, per_neuron=True, seed=42, verbose=True,
) -> Dict[str, PhaseResult]:
    """Run the full 3-phase experiment. Returns dict with keys 'phase1', 'phase2', 'phase3'."""
    if hidden_dims is None:
        hidden_dims = [256, 128]

    np.random.seed(seed)
    input_dim = X_train.shape[1]
    output_dim = len(np.unique(y_train))

    if verbose:
        print(f"  Architecture: {[input_dim] + hidden_dims + [output_dim]}")

    trainer = MLPTrainer(
        epochs=weight_epochs, batch_size=batch_size, learning_rate=weight_lr,
        early_stopping_patience=patience, verbose=verbose, print_every=10,
    )

    results: Dict[str, PhaseResult] = {}

    # ── PHASE 1: Train weights with fixed ReLU ──
    if verbose:
        print(f"\n{'─'*70}")
        print("  PHASE 1: Train weights with fixed ReLU activation")
        print(f"{'─'*70}")

    np.random.seed(seed)
    phase1_model = build_fixed_relu_model(input_dim, hidden_dims, output_dim, weight_lr)
    phase1_init_weights = extract_all_weights(phase1_model)
    h1 = train_weights_only(phase1_model, X_train, y_train, X_val, y_val, trainer)

    results['phase1'] = PhaseResult(
        name="Phase 1: Weights on Fixed ReLU",
        epochs_trained=h1.epochs_trained, training_time=h1.training_time,
        final_train_acc=h1.train_accuracies[-1], final_val_acc=h1.val_accuracies[-1],
        best_val_acc=max(h1.val_accuracies),
        train_losses=h1.train_losses, val_losses=h1.val_losses,
    )

    if verbose:
        print(f"  -> Epochs: {h1.epochs_trained} | Time: {h1.training_time:.2f}s")
        print(f"  -> Train acc: {h1.train_accuracies[-1]:.4f} | Val acc: {h1.val_accuracies[-1]:.4f}")

    # ── PHASE 2: Learn activations on pretrained weights (weights frozen) ──
    if verbose:
        print(f"\n{'─'*70}")
        print("  PHASE 2: Train DynamicReLU(a,b) on pretrained weights (W frozen)")
        print(f"{'─'*70}")

    phase2_model = build_dynamic_relu_model(
        input_dim, hidden_dims, output_dim,
        lr=weight_lr, activation_lr=activation_lr, per_neuron=per_neuron,
    )
    copy_weights_into(phase2_model, phase1_model)

    pre_acc = float(trainer._compute_accuracy(phase2_model, X_val, y_val))
    if verbose:
        print(f"  Accuracy before activation training (should match Phase 1): {pre_acc:.4f}")

    h2 = train_activations_only(
        phase2_model, X_train, y_train, X_val, y_val,
        trainer, epochs=activation_epochs,
    )

    results['phase2'] = PhaseResult(
        name="Phase 2: Learn activations (weights frozen)",
        epochs_trained=h2.epochs_trained, training_time=h2.training_time,
        final_train_acc=h2.train_accuracies[-1], final_val_acc=h2.val_accuracies[-1],
        best_val_acc=max(h2.val_accuracies),
        train_losses=h2.train_losses, val_losses=h2.val_losses,
    )

    if verbose:
        print(f"  -> Epochs: {h2.epochs_trained} | Time: {h2.training_time:.2f}s")
        print(f"  -> Val acc after activation training: {h2.val_accuracies[-1]:.4f}")

    pretrained_act_params = extract_activation_params(phase2_model)
    if verbose:
        for i, (a, b) in enumerate(pretrained_act_params):
            if a.size > 1:
                print(f"  Layer {i}: a in [{a.min():.4f}, {a.max():.4f}] (mean {a.mean():.4f}), "
                      f"b in [{b.min():.4f}, {b.max():.4f}] (mean {b.mean():.4f})")
            else:
                print(f"  Layer {i}: a={float(a):.4f}, b={float(b):.4f}")

    # ── PHASE 3: Train random weights with pretrained activations (a,b frozen) ──
    if verbose:
        print(f"\n{'─'*70}")
        print("  PHASE 3: Train random weights with pretrained DynamicReLU(a*,b*)")
        print(f"{'─'*70}")

    np.random.seed(seed)
    phase3_model = build_dynamic_relu_model(
        input_dim, hidden_dims, output_dim,
        lr=weight_lr, activation_lr=activation_lr, per_neuron=per_neuron,
    )
    inject_activation_params(phase3_model, pretrained_act_params)

    phase3_init_weights = extract_all_weights(phase3_model)
    init_cos = cosine_similarity(phase1_init_weights, phase3_init_weights)
    if verbose:
        print(f"  Initial weight cosine similarity with Phase 1 init: {init_cos:.6f}")

    h3 = train_weights_only(phase3_model, X_train, y_train, X_val, y_val, trainer)

    results['phase3'] = PhaseResult(
        name="Phase 3: Random weights + pretrained activations",
        epochs_trained=h3.epochs_trained, training_time=h3.training_time,
        final_train_acc=h3.train_accuracies[-1], final_val_acc=h3.val_accuracies[-1],
        best_val_acc=max(h3.val_accuracies),
        train_losses=h3.train_losses, val_losses=h3.val_losses,
    )

    if verbose:
        print(f"  -> Epochs: {h3.epochs_trained} | Time: {h3.training_time:.2f}s")
        print(f"  -> Train acc: {h3.train_accuracies[-1]:.4f} | Val acc: {h3.val_accuracies[-1]:.4f}")

    return results

## Multi-Seed Suite Runner

In [ ]:
def run_experiment_suite(
    dataset_type, hidden_dims, epochs, activation_epochs,
    batch_size, learning_rate, seeds, patience=10, per_neuron=True,
) -> Dict[str, AggregatedPhaseResult]:
    """Run the 3-phase experiment with multiple seeds on one dataset."""
    print(f"Loading {dataset_type} dataset...")
    config = DatasetConfig(dataset_type=dataset_type, random_state=seeds[0])
    dm = DataManager(config)
    X_train, X_val, y_train, y_val = dm.generate_dataset()
    print(f"  Training: {len(X_train):,} | Test: {len(X_val):,} | "
          f"Features: {X_train.shape[1]} | Classes: {len(np.unique(y_train))}")

    agg = {
        "Phase 1 (Fixed ReLU)": AggregatedPhaseResult("Phase 1 (Fixed ReLU)"),
        "Phase 2 (Activation Training)": AggregatedPhaseResult("Phase 2 (Activation Training)"),
        "Phase 3 (Pretrained Act.)": AggregatedPhaseResult("Phase 3 (Pretrained Act.)"),
    }

    for i, seed in enumerate(seeds):
        print(f"\n  Seed {i+1}/{len(seeds)} (seed={seed})...")

        results = run_experiment(
            X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
            hidden_dims=hidden_dims, weight_lr=learning_rate,
            activation_lr=learning_rate, weight_epochs=epochs,
            activation_epochs=activation_epochs, batch_size=batch_size,
            patience=patience, per_neuron=per_neuron, seed=seed, verbose=False,
        )

        for phase_key, agg_key in [
            ('phase1', "Phase 1 (Fixed ReLU)"),
            ('phase2', "Phase 2 (Activation Training)"),
            ('phase3', "Phase 3 (Pretrained Act.)"),
        ]:
            p = results[phase_key]
            agg[agg_key].epochs.append(p.epochs_trained)
            agg[agg_key].times.append(p.training_time)
            agg[agg_key].val_accs.append(p.final_val_acc)
            agg[agg_key].best_val_accs.append(p.best_val_acc)
            agg[agg_key].train_accs.append(p.final_train_acc)

        p1, p3 = results['phase1'], results['phase3']
        print(f"    P1: {p1.epochs_trained} ep, {p1.training_time:.1f}s, "
              f"val={p1.final_val_acc:.4f}  |  "
              f"P3: {p3.epochs_trained} ep, {p3.training_time:.1f}s, "
              f"val={p3.final_val_acc:.4f}  "
              f"(delta={p3.final_val_acc - p1.final_val_acc:+.4f})")

    return agg

## Run Experiments on All Datasets

In [ ]:
all_results = {}

for dataset in DATASETS:
    print(f"\n{'#'*100}")
    print(f"  DATASET: {dataset.upper()}")
    print(f"{'#'*100}")

    agg = run_experiment_suite(
        dataset_type=dataset, hidden_dims=HIDDEN_DIMS,
        epochs=EPOCHS, activation_epochs=ACTIVATION_EPOCHS,
        batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
        seeds=SEEDS, patience=PATIENCE, per_neuron=True,
    )
    all_results[dataset] = agg

## Results Summary

In [ ]:
for dataset_name, agg in all_results.items():
    print("\n" + "=" * 100)
    print(f"EXPERIMENT RESULTS: {dataset_name.upper()} ({N_SEEDS} seeds)")
    print("=" * 100)

    headers = ["Phase", "Train Acc", "Test Acc", "Time (s)", "Epochs"]
    row_fmt = "{:<45} {:>18} {:>18} {:>10} {:>8}"
    print(row_fmt.format(*headers))
    print("-" * 100)

    for name, r in agg.items():
        print(row_fmt.format(
            name[:45],
            f"{r.train_acc_mean:.4f} +/- {r.train_acc_std:.4f}",
            f"{r.val_acc_mean:.4f} +/- {r.val_acc_std:.4f}",
            f"{r.time_mean:.2f}",
            f"{r.epochs_mean:.1f}",
        ))
    print("-" * 100)

    p1 = agg["Phase 1 (Fixed ReLU)"]
    p3 = agg["Phase 3 (Pretrained Act.)"]
    acc_diff = p3.val_acc_mean - p1.val_acc_mean
    epoch_diff = p1.epochs_mean - p3.epochs_mean

    print(f"\n  Phase 1 vs Phase 3:")
    print(f"    Accuracy delta:  {acc_diff:+.4f}")
    print(f"    Epoch delta:     {epoch_diff:+.1f} epochs saved")

    # Paired t-test
    t_ep, p_ep = stats.ttest_rel(p1.epochs, p3.epochs)
    t_acc, p_acc = stats.ttest_rel(p1.val_accs, p3.val_accs)
    print(f"    Epochs  t-test: t={t_ep:.3f}, p={p_ep:.4f} {'*' if p_ep < 0.05 else ''}")
    print(f"    Acc     t-test: t={t_acc:.3f}, p={p_acc:.4f} {'*' if p_acc < 0.05 else ''}")

## Final Summary & Research Answers

In [ ]:
print("=" * 100)
print(f"FINAL SUMMARY (Mean +/- Std over {N_SEEDS} seeds)")
print("=" * 100)

all_acc_diffs = []
all_epoch_diffs = []

for dataset_name, agg in all_results.items():
    p1 = agg["Phase 1 (Fixed ReLU)"]
    p3 = agg["Phase 3 (Pretrained Act.)"]
    acc_diff = p3.val_acc_mean - p1.val_acc_mean
    epoch_diff = p1.epochs_mean - p3.epochs_mean
    all_acc_diffs.append(acc_diff)
    all_epoch_diffs.append(epoch_diff)

    print(f"\n{dataset_name.upper()}:")
    print(f"   Phase 1 (Fixed ReLU):      {p1.val_acc_mean:.4f} +/- {p1.val_acc_std:.4f}  ({p1.epochs_mean:.1f} epochs)")
    print(f"   Phase 3 (Pretrained Act.):  {p3.val_acc_mean:.4f} +/- {p3.val_acc_std:.4f}  ({p3.epochs_mean:.1f} epochs)")
    print(f"   Accuracy delta: {acc_diff:+.4f} ({acc_diff*100:+.2f}%)")
    print(f"   Epoch delta:    {epoch_diff:+.1f} epochs")

mean_acc_diff = float(np.mean(all_acc_diffs))
mean_epoch_diff = float(np.mean(all_epoch_diffs))

print(f"\n{'='*100}")
print("ANSWERS TO RESEARCH QUESTIONS")
print(f"{'='*100}")

print(f"\n  Q1: Will weights be different?")
print(f"      Weights are expected to be nearly identical (same loss landscape")
print(f"      with negligible activation perturbation from DynamicReLU(a~0, b~1)).")

print(f"\n  Q2: Did weight training finish earlier with pretrained activations?")
if mean_epoch_diff > 1:
    print(f"      YES, on average {mean_epoch_diff:.1f} fewer epochs across datasets.")
elif mean_epoch_diff < -1:
    print(f"      NO, pretrained activations took {-mean_epoch_diff:.1f} MORE epochs.")
else:
    print(f"      NO significant difference ({mean_epoch_diff:+.1f} epochs on average).")

print(f"\n  Overall accuracy change: {mean_acc_diff:+.4f} (averaged across datasets).")

## Save Results to CSV

In [ ]:
rows = []
for dataset_name, agg in all_results.items():
    for phase_name, r in agg.items():
        rows.append({
            "dataset": dataset_name,
            "phase": phase_name,
            "n_seeds": N_SEEDS,
            "train_acc_mean": r.train_acc_mean,
            "train_acc_std": r.train_acc_std,
            "test_acc_mean": r.val_acc_mean,
            "test_acc_std": r.val_acc_std,
            "epochs_mean": r.epochs_mean,
            "epochs_std": r.epochs_std,
            "time_mean": r.time_mean,
            "time_std": r.time_std,
            "timestamp": datetime.now().isoformat(),
        })

df = pd.DataFrame(rows)
df.to_csv("pretrained_activation_results.csv", index=False)
print(f"Results saved to: pretrained_activation_results.csv")
df